<a href="https://colab.research.google.com/github/saqib0-cpu/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:

!pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib


In [4]:
# Cell 2: HF Token
from getpass import getpass
hf_token = getpass("HF token daalo: ")


HF token daalo: ··········


In [5]:
import duckdb
con = duckdb.connect()

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

In [7]:
# Cell 4: Define rel FIRST — ye missing tha
rel = "hf://datasets/FlyRank/internship-warehouse"
# Cell 5: Ab glob() chalega kyunki rel define ho chuka hai
files = con.execute(f"""
    SELECT * FROM glob('{rel}/fact_content_daily_performance/**')
""").df()

In [8]:

import pandas as pd
pd.set_option('display.max_colwidth', None)
print(files.to_string())



                                                                                                      file
0   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
1   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
2   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
3   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
4   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
5   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
6   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
7   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08/data_0.parquet
8   hf://datasets/FlyRank/internship-

In [10]:
#Cell 6: features_df banane wala main query
rel_daily = f"{rel}/fact_content_daily_performance/month=*/data_0.parquet"

query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    SUM(CASE WHEN month BETWEEN '2025-08' AND '2025-10' THEN gsc_clicks ELSE 0 END) AS clicks_a,
    SUM(CASE WHEN month BETWEEN '2025-08' AND '2025-10' THEN gsc_impressions ELSE 0 END) AS impr_a,
    AVG(CASE WHEN month BETWEEN '2025-08' AND '2025-10' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_a,
    SUM(CASE WHEN month BETWEEN '2025-11' AND '2026-01' THEN gsc_clicks ELSE 0 END) AS clicks_b,
    SUM(CASE WHEN month BETWEEN '2025-11' AND '2026-01' THEN gsc_impressions ELSE 0 END) AS impr_b,
    AVG(CASE WHEN month BETWEEN '2025-11' AND '2026-01' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_b,
    SUM(CASE WHEN month BETWEEN '2026-02' AND '2026-04' THEN gsc_clicks ELSE 0 END) AS clicks_c,
    SUM(CASE WHEN month BETWEEN '2025-11' AND '2026-01' THEN ga4_sessions ELSE 0 END) AS sessions_b,
    SUM(CASE WHEN month BETWEEN '2025-11' AND '2026-01' THEN sessions_ai ELSE 0 END) AS ai_sessions_b,
    SUM(CASE WHEN month BETWEEN '2025-11' AND '2026-01' THEN scroll_events ELSE 0 END) AS scroll_b
FROM read_parquet('{rel_daily}')
WHERE gsc_data_available = True
GROUP BY content_hash_id, client_hash_id
HAVING impr_a > 0 OR impr_b > 0
"""

features_df = con.execute(query).df()
print(features_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(146651, 12)


In [11]:
content_query = f"""
SELECT content_hash_id, content_type, search_volume, competition_level,
       main_intent, word_count, is_published,
       content_created_date, last_optimized_date
FROM read_parquet('{rel}/dim_content.parquet')
WHERE is_deleted = False AND is_published = True
"""

df_content = con.execute(content_query).df()
features_df = features_df.merge(df_content, on='content_hash_id', how='left')
print(features_df.shape)
print(features_df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(146651, 20)
['content_hash_id', 'client_hash_id', 'clicks_a', 'impr_a', 'pos_a', 'clicks_b', 'impr_b', 'pos_b', 'clicks_c', 'sessions_b', 'ai_sessions_b', 'scroll_b', 'content_type', 'search_volume', 'competition_level', 'main_intent', 'word_count', 'is_published', 'content_created_date', 'last_optimized_date']


In [12]:
import numpy as np

features_df['clicks_delta_pct'] = np.where(
    features_df['clicks_a'] > 0,
    (features_df['clicks_b'] - features_df['clicks_a']) / features_df['clicks_a'],
    np.where(features_df['clicks_b'] > 0, 1.0, 0.0)
)
features_df['position_delta'] = features_df['pos_a'] - features_df['pos_b']
features_df['ctr_b'] = np.where(features_df['impr_b'] > 0, features_df['clicks_b'] / features_df['impr_b'], 0)
features_df['ctr_a'] = np.where(features_df['impr_a'] > 0, features_df['clicks_a'] / features_df['impr_a'], 0)
features_df['future_growth'] = np.where(
    features_df['clicks_b'] > 0,
    (features_df['clicks_c'] - features_df['clicks_b']) / features_df['clicks_b'], 0
)
features_df['will_decline'] = (features_df['future_growth'] < -0.15).astype(int)
features_df = features_df.fillna(0)

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score
import pandas as pd

feature_cols = ['clicks_delta_pct', 'position_delta', 'ctr_b', 'ctr_a',
                 'search_volume', 'competition_level', 'word_count']

features_df['competition_level'] = features_df['competition_level'].astype('category').cat.codes

X = features_df[feature_cols]
y = features_df['will_decline']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [15]:
def classify(row):
    if row['clicks_delta_pct'] > 0.2 and row['position_delta'] > 0:
        return 'growing'
    elif row['clicks_delta_pct'] < -0.2 and row['position_delta'] < 0:
        return 'declining'
    elif row['clicks_delta_pct'] > 0.1 and row['position_delta'] < 0:
        return 'recovering'
    elif abs(row['clicks_delta_pct']) <= 0.1:
        return 'stagnant'
    else:
        return 'volatile'

features_df['status'] = features_df.apply(classify, axis=1)

action_map = {
    'growing': 'protect',
    'declining': 'rewrite',
    'recovering': 'monitor',
    'stagnant': 'improve',
    'volatile': 'review'
}
features_df['action'] = features_df['status'].map(action_map)

print(features_df['status'].value_counts())

status
stagnant      88025
volatile      29217
growing       15444
recovering     9466
declining      4499
Name: count, dtype: int64


In [16]:
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(classification_report(y_test, preds))

baseline_preds = (features_df.loc[X_test.index, 'status'] == 'declining').astype(int)
model_f1 = f1_score(y_test, preds)
baseline_f1 = f1_score(y_test, baseline_preds)

results = pd.DataFrame({
    'Model': ['Rule-Based Baseline (Week 4)', 'Gradient Boosting'],
    'F1': [baseline_f1, model_f1]
})
print(results)

              precision    recall  f1-score   support

           0       0.91      0.96      0.93     24482
           1       0.70      0.52      0.60      4849

    accuracy                           0.88     29331
   macro avg       0.80      0.74      0.76     29331
weighted avg       0.87      0.88      0.88     29331

                          Model        F1
0  Rule-Based Baseline (Week 4)  0.156266
1             Gradient Boosting  0.595720


In [18]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=features_df['client_hash_id']))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(features_df.iloc[train_idx]['client_hash_id'])
test_clients = set(features_df.iloc[test_idx]['client_hash_id'])
print("Client overlap between train/test:", len(train_clients & test_clients))

model_grouped = GradientBoostingClassifier(random_state=42)
model_grouped.fit(X_train_g, y_train_g)
preds_g = model_grouped.predict(X_test_g)

print(classification_report(y_test_g, preds_g))
grouped_f1 = f1_score(y_test_g, preds_g)

print(f"Before (random split, Week 5): F1 = 0.598")
print(f"After (client-grouped split): F1 = {grouped_f1:.3f}")

Client overlap between train/test: 0
              precision    recall  f1-score   support

           0       0.89      0.89      0.89     33358
           1       0.51      0.51      0.51      7129

    accuracy                           0.83     40487
   macro avg       0.70      0.70      0.70     40487
weighted avg       0.83      0.83      0.83     40487

Before (random split, Week 5): F1 = 0.598
After (client-grouped split): F1 = 0.507


In [19]:
import pandas as pd

# Leakage notes
leakage_notes = {
    'clicks_delta_pct': 'Built only from Windows A and B — before the label window. Safe.',
    'position_delta': 'Built only from Windows A and B. Safe.',
    'ctr_b': 'Built from Window B, which precedes the label window (Window C). No direct overlap.',
    'ctr_a': 'Built from Window A. Safe.',
    'search_volume': 'Static content metadata — need to confirm it was not updated after Window C.',
    'competition_level': 'Static metadata — same caveat as search_volume.',
    'word_count': 'Current snapshot — could leak if the content was optimized in response to the decline.',
}

for f, note in leakage_notes.items():
    print(f"{f}: {note}")

# Convert optimization date to datetime
features_df['last_optimized_date'] = pd.to_datetime(
    features_df['last_optimized_date'],
    errors='coerce'
)

# Window C: 2026-02-01 through 2026-04-30
updated_during_label_window = features_df[
    (features_df['last_optimized_date'] >= '2026-02-01') &
    (features_df['last_optimized_date'] <= '2026-04-30')
]

print(
    f"\nPages optimized during label window C: "
    f"{len(updated_during_label_window)} of {len(features_df)}"
)

clicks_delta_pct: Built only from Windows A and B — before the label window. Safe.
position_delta: Built only from Windows A and B. Safe.
ctr_b: Built from Window B, which precedes the label window (Window C). No direct overlap.
ctr_a: Built from Window A. Safe.
search_volume: Static content metadata — need to confirm it was not updated after Window C.
competition_level: Static metadata — same caveat as search_volume.
word_count: Current snapshot — could leak if the content was optimized in response to the decline.

Pages optimized during label window C: 494 of 146651


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [20]:
import pandas as pd

# reason code generator
def reason_code(row):
    if row['status'] == 'declining':
        return f"CTR/clicks dropped {abs(row['clicks_delta_pct']):.0%}, position worsened by {abs(row['position_delta']):.1f}"
    elif row['status'] == 'growing':
        return f"Clicks up {row['clicks_delta_pct']:.0%}, position improved by {row['position_delta']:.1f}"
    elif row['status'] == 'recovering':
        return f"Clicks up {row['clicks_delta_pct']:.0%} despite weaker position — organic recovery signal"
    elif row['status'] == 'stagnant':
        return f"Click change within ±10% — low CTR ({row['ctr_b']:.2%}) suggests metadata gap"
    else:
        return "Mixed/noisy signal — needs manual review"

features_df['reason_code'] = features_df.apply(reason_code, axis=1)

# use the model's predicted decline probability to rank urgency
features_df['decline_risk_score'] = model_grouped.predict_proba(X[feature_cols])[:, 1]

ranked_queue = features_df[[
    'content_hash_id', 'status', 'action', 'reason_code',
    'decline_risk_score', 'clicks_delta_pct', 'position_delta', 'ctr_b'
]].sort_values('decline_risk_score', ascending=False)

ranked_queue.head(20)

,content_hash_id,status,action,reason_code,decline_risk_score,clicks_delta_pct,position_delta,ctr_b
15657,content_671d56406b8caf73,growing,protect,"Clicks up 100%, position improved by 11.2",0.976809,1.000000,11.239583,1.000000
59989,content_ed8e0cf5d941b6dc,recovering,monitor,Clicks up 100% despite weaker position — organic recovery signal,0.969208,1.000000,-17.333333,0.333333
8032,content_94381e653a1279d1,declining,rewrite,"CTR/clicks dropped 60%, position worsened by 19.1",0.969136,-0.600000,-19.120092,0.011527
8154,content_ef9f50431bd76160,declining,rewrite,"CTR/clicks dropped 57%, position worsened by 27.2",0.965582,-0.571429,-27.217467,0.008000
5489,content_aa1bc194cb8fc3ef,declining,rewrite,"CTR/clicks dropped 62%, position worsened by 21.6",0.965502,-0.625000,-21.557294,0.008287
8430,content_e301d52db1805035,declining,rewrite,"CTR/clicks dropped 54%, position worsened by 11.9",0.962477,-0.538462,-11.868623,0.011719
30147,content_67c4ae1087f1c0eb,growing,protect,"Clicks up 100%, position improved by 37.2",0.961990,1.000000,37.250000,0.400000
95541,content_88b81af73dc5079c,recovering,monitor,Clicks up 100% despite weaker position — organic recovery signal,0.959948,1.000000,-4.000000,0.500000
4020,content_b6d49be40fcf98ca,declining,rewrite,"CTR/clicks dropped 60%, position worsened by 10.5",0.959872,-0.595745,-10.503053,0.015574
8576,content_4bebf56d8f5c9593,declining,rewrite,"CTR/clicks dropped 65%, position worsened by 24.3",0.959827,-0.647059,-24.321380,0.011407


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
Intended use: This playbook ranks content pages by estimated decline risk and suggests
an action (protect, rewrite, monitor, improve, review) to help a content team prioritize
where to spend manual review time first — it does not replace editorial judgment.

Limits:
- Trained and validated on FlyRank internship warehouse data (Aug 2025–Apr 2026) —
  not guaranteed to generalize to other time periods, verticals, or clients outside
  this dataset.
- The will_decline label uses a 15% click-drop threshold — a design choice, not a
  validated business rule.
- ctr_b dominates the model's decisions (~78% importance) — the ranking is largely a
  CTR-efficiency signal, not a holistic content-quality judgment.
- Measured F1 under a client-grouped split was [insert Week-6 number] — lower than the
  original random-split score, meaning real-world generalization is likely more modest
  than the headline number suggests.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*
## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*
Human review required for:
- Any "rewrite" action before content is actually rewritten or retired
- Pages flagged "declining" with low search_volume (model has less reliable signal here)
- Any client-facing or legally sensitive content, regardless of score

NOT to be automated (no-go list):
- Auto-deleting or auto-unpublishing content based on decline_risk_score
- Auto-generating replacement content without human edit/approval
- Using this score as the sole input for client billing or performance conversations
- Applying this model to a clie


Monitoring:
- Track weekly what % of "declining" flagged pages actually saw continued click drops
  in the following window — measures real precision over time
- Track feature importance drift — if ctr_b's dominance changes significantly, the
  underlying pattern may have shifted

Retrain triggers:
- If observed precision on live data drops meaningfully below the measured 0.70 baseline
- Every quarter, regardless of performance, since search behavior/algorithms shift
- If a new content type or client vertical is added that wasn't in training data

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [21]:
import os
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

ranked_queue.to_csv('work/outputs/ranked_action_queue.csv', index=False)

import json
metrics = {
    'baseline_f1_random_split': 0.154,
    'model_f1_random_split': 0.598,
    'model_f1_grouped_split': None,  # apna Week-6 actual number daalo
    'precision': 0.70,
    'recall': 0.52,
    'top_feature': 'ctr_b',
    'top_feature_importance': 0.779
}
with open('work/outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported ranked_action_queue.csv and metrics.json")

Exported ranked_action_queue.csv and metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ *] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.